# Edu Nexus – Phase 1: File Scanner & Metadata Index

This notebook scans the local edu_nexus_db raw data folders
and builds a metadata index of all available files.

Responsibilities:
- Detect PDFs, PPTs, images, and other files
- Store file paths, types, and source category
- Output a structured file_index.json for downstream pipelines

No content extraction is done in this notebook.


In [1]:
# Standard library imports
import os
import json
from pathlib import Path
from datetime import datetime


In [2]:
# ===== CONFIGURATION =====

# Change this ONLY if folder location changes
BASE_DB_PATH = Path("../../edu_nexus_db")

RAW_PATH = BASE_DB_PATH / "raw"
METADATA_PATH = BASE_DB_PATH / "metadata"

# Output file
INDEX_FILE = METADATA_PATH / "file_index.json"

print("Base DB Path:", BASE_DB_PATH.resolve())
print("Raw Data Path:", RAW_PATH.resolve())


Base DB Path: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db
Raw Data Path: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db\raw


In [3]:
# ===== VALIDATION =====

required_folders = [
    RAW_PATH / "pdf",
    RAW_PATH / "ppt",
    RAW_PATH / "images",
    RAW_PATH / "other",
    METADATA_PATH
]

missing_folders = [str(p) for p in required_folders if not p.exists()]

if missing_folders:
    raise FileNotFoundError(f"Missing required folders: {missing_folders}")

print("All required folders are present ✅")


All required folders are present ✅


In [4]:
# ===== FILE SCANNER =====

SUPPORTED_TYPES = {
    "pdf": [".pdf"],
    "ppt": [".ppt", ".pptx"],
    "images": [".png", ".jpg", ".jpeg", ".tiff"],
    "other": [".txt", ".docx"]
}

file_index = []
file_id = 1

for category, extensions in SUPPORTED_TYPES.items():
    category_path = RAW_PATH / category
    
    for root, _, files in os.walk(category_path):
        for file in files:
            file_path = Path(root) / file
            ext = file_path.suffix.lower()
            
            if ext in extensions:
                file_index.append({
                    "id": file_id,
                    "filename": file_path.name,
                    "extension": ext,
                    "category": category,
                    "absolute_path": str(file_path.resolve()),
                    "status": "raw",
                    "added_on": datetime.utcnow().isoformat()
                })
                file_id += 1

print(f"Total files indexed: {len(file_index)}")


Total files indexed: 0


In [5]:
# ===== SAVE METADATA INDEX =====

METADATA_PATH.mkdir(parents=True, exist_ok=True)

with open(INDEX_FILE, "w", encoding="utf-8") as f:
    json.dump(file_index, f, indent=4)

print(f"Metadata index saved to: {INDEX_FILE.resolve()}")


Metadata index saved to: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db\metadata\file_index.json
